# Vehicle detection — training (YOLOv8)

Notebook gốc từ Kaggle: tiền xử lý dataset, augment class hiếm, train và validate model dùng cho backend (`../weights/best_v6.pt`).

**Cấu trúc thư mục**
- `dataset/` — đặt dataset YOLO đã chuẩn bị (xem `data.yaml.example`)
- `work/` — output tạm khi chạy local (remap, merge, augment…)
- `runs/` — kết quả train Ultralytics

**Sau khi train:** copy `runs/.../weights/best.pt` → `../weights/best_v6.pt`

> Các cell dùng đường dẫn `/kaggle/...` giữ nguyên cho Kaggle; khi chạy local hãy đổi sang `training/dataset/` hoặc `training/work/`.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install ultralytics

In [ ]:

from ultralytics import YOLO
import os

#model = YOLO("/kaggle/input/models/phandat123/modelv1/pytorch/default/1/best_v1m.pt")



In [ ]:
#thống kê datasets
import os
from collections import defaultdict

# ====== CONFIG ======
labels_path = "/kaggle/working/train/labels"  # sửa lại path của bạn
class_names = {
    0: "car",
    1: "bus",
    2: "truck",
    3: "motorcycle",
    4: "bicycle",
    5: "person"
}
# ====================

class_counts = defaultdict(int)
total_objects = 0

# Duyệt tất cả file label
for file in os.listdir(labels_path):
    if file.endswith(".txt"):
        with open(os.path.join(labels_path, file), "r") as f:
            lines = f.readlines()
            
            for line in lines:
                if line.strip() == "":
                    continue
                class_id = int(line.split()[0])
                class_counts[class_id] += 1
                total_objects += 1

# ====== In kết quả ======
print(f"Total objects: {total_objects}\n")

print("Class distribution:")
for class_id, count in class_counts.items():
    percent = (count / total_objects) * 100
    name = class_names.get(class_id, f"class_{class_id}")
    
    print(f"{name:15}: {count:6} ({percent:.2f}%)")

In [ ]:
#xem ảnh mẫu
import cv2
import matplotlib.pyplot as plt
import os

def check_single_bbox(image_path, label_path=None):
    # Nếu không truyền label_path, tự động tìm file .txt cùng tên với ảnh
    if label_path is None:
        label_path = image_path.rsplit('.', 1)[0] + '.txt'

    if not os.path.exists(image_path):
        print(f"Lỗi: Không tìm thấy ảnh tại {image_path}")
        return

    # Đọc ảnh
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape

    # Đọc nhãn (YOLO format)
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f.readlines():
                data = line.split()
                if len(data) < 5: continue
                
                cls, x, y, bw, bh = map(float, data)

                # Chuyển đổi từ YOLO format (0-1) sang tọa độ Pixel
                # Công thức: x_center * width, y_center * height ...
                x1 = int((x - bw/2) * w)
                y1 = int((y - bh/2) * h)
                x2 = int((x + bw/2) * w)
                y2 = int((y + bh/2) * h)

                # Vẽ hình chữ nhật (màu đỏ, độ dày 2)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                # Thêm tên class nếu muốn
                cv2.putText(img, f"Class: {int(cls)}", (x1, y1 - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    else:
        print(f"Cảnh báo: Không tìm thấy file nhãn tại {label_path}")

    # Hiển thị kết quả
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.title(os.path.basename(image_path))
    plt.axis("off")
    plt.show()

# --- SỬ DỤNG TẠI ĐÂY ---
img_ptr = "/kaggle/input/datasets/phandat123/my-vehicle-detection-datasets-v2/final_traffic_dataset/test/images/test_20623.jpg"
lbl_ptr = "/kaggle/input/datasets/phandat123/my-vehicle-detection-datasets-v2/final_traffic_dataset/test/labels/test_20623.txt"

check_single_bbox(img_ptr, lbl_ptr)

In [ ]:
#check miss labels
import os

DATASET_PATH = "/kaggle/input/datasets/pkdarabi/vehicle-detection-image-dataset/Apply_Grayscale/Apply_Grayscale/Vehicles_Detection.v9i.yolov8"

image_dir = f"{DATASET_PATH}/train/images"
label_dir = f"{DATASET_PATH}/train/labels"

images = os.listdir(image_dir)

missing_labels = []
bad_boxes = []

for img in images:
    
    label_file = os.path.join(label_dir, img.rsplit(".",1)[0] + ".txt")
    
    if not os.path.exists(label_file):
        missing_labels.append(img)
        continue
    
    with open(label_file) as f:
        for line in f:
            parts = line.split()
            
            if len(parts) != 5:
                bad_boxes.append((img,"format_error"))
                continue
                
            cls, x, y, w, h = map(float, parts)
            
            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                bad_boxes.append((img,"bbox_out_of_range"))

print("Total images:", len(images))
print("Missing labels:", len(missing_labels))
print("Bad boxes:", len(bad_boxes))

print("\nExample missing labels:", missing_labels[:5])
print("Example bad boxes:", bad_boxes[:5])

In [ ]:
# remap labels
import os
import shutil

# mapping cho từng dataset
map_dataset1 ={
    0:3,
    1:0,
}
map_dataset2 = {
    0: 4,  
    1: 1,
    2:0,
    3:3
}

map_dataset3 = {
    0: 4,
    1: 1,
    2: 0,
    3: 3,
    4: 5,
    5: 2
}
map_dataset4 = {
    0: 2,
}

# Lưu ý: Chỉ để đường dẫn gốc của dataset, không để đến tận thư mục /train
datasets = [
    ("/kaggle/input/datasets/phandat123/add-dataset", map_dataset1),
    ("/kaggle/input/datasets/phandat123/add-datasets2/Intersection Traffic.v9-normal-data.yolov8", map_dataset2),
    ("/kaggle/input/datasets/phandat123/add-few-class/Detect.v1i.yolov8", map_dataset3),
    ("/kaggle/input/datasets/phandat123/add-few-class/truck detection.v1i.yolov8", map_dataset4),
]

def copy_and_remap(dataset_root, mapping, dataset_id):
    # Các thư mục con thường gặp trong YOLO
    subsets = ['train', 'valid', 'test']
    
    for subset in subsets:
        src_subset_path = os.path.join(dataset_root, subset)
        
        # Kiểm tra xem thư mục subset (train/valid/test) có tồn tại không
        if not os.path.exists(src_subset_path):
            print(f"Bỏ qua: Không tìm thấy thư mục {subset} tại {dataset_root}")
            continue
            
        dst_folder = f"/kaggle/working/dataset_{dataset_id}/{subset}"
        
        images_src = os.path.join(src_subset_path, "images")
        labels_src = os.path.join(src_subset_path, "labels")

        images_dst = os.path.join(dst_folder, "images")
        labels_dst = os.path.join(dst_folder, "labels")

        os.makedirs(images_dst, exist_ok=True)
        os.makedirs(labels_dst, exist_ok=True)

        # 1. Copy images
        if os.path.exists(images_src):
            for img in os.listdir(images_src):
                shutil.copy(os.path.join(images_src, img), images_dst)
        
        # 2. Copy + Remap labels
        if os.path.exists(labels_src):
            for file in os.listdir(labels_src):
                if not file.endswith(".txt"):
                    continue

                src_path = os.path.join(labels_src, file)
                dst_path = os.path.join(labels_dst, file)

                with open(src_path, 'r') as f:
                    lines = f.readlines()

                new_lines = []
                for line in lines:
                    parts = line.split()
                    if len(parts) == 0: continue
                    
                    old_id = int(parts[0])
                    # Chỉ thay đổi nếu ID có trong mapping, nếu không thì giữ nguyên (hoặc tùy chỉnh)
                    if old_id in mapping:
                        parts[0] = str(mapping[old_id])
                    
                    new_lines.append(" ".join(parts))

                with open(dst_path, "w") as f:
                    f.write("\n".join(new_lines))
        
        print(f"--- Đã xong {subset} cho Dataset {dataset_id}")

# Chạy vòng lặp chính
for i, (folder, mapping) in enumerate(datasets):
    copy_and_remap(folder, mapping, i)

print("Hoàn tất: Đã copy và remap cho tất cả các folder train/valid/test!")


In [ ]:
!ls /kaggle/input/

In [ ]:
#lọc lấy class nhỏ
import os
import shutil
from pathlib import Path

# --- CẤU HÌNH ---
# Danh sách các đường dẫn dataset nguồn (Kaggle/Colab path)
DATASET_SOURCES = [
    '/kaggle/working/dataset_0',
    '/kaggle/working/dataset_1',
]

OUTPUT_ROOT = '/kaggle/working/merged_rare_dataset'
RARE_CLASS_IDS = ['1', '2', '5','4']  # ID của truck, bus, bicycle

SUBSETS = ['train', 'valid', 'test']

def merge_and_filter_datasets():
    # Tạo cấu trúc thư mục đích
    for subset in SUBSETS:
        os.makedirs(os.path.join(OUTPUT_ROOT, subset, 'images'), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_ROOT, subset, 'labels'), exist_ok=True)

    print(f"🚀 Bắt đầu lọc và gộp {len(DATASET_SOURCES)} nguồn dữ liệu...")

    for idx, source_path in enumerate(DATASET_SOURCES):
        dataset_name = os.path.basename(source_path)
        print(f"\n📂 Đang xử lý: {dataset_name}")

        for subset in SUBSETS:
            src_label_dir = os.path.join(source_path, subset, 'labels')
            src_image_dir = os.path.join(source_path, subset, 'images')

            if not os.path.exists(src_label_dir):
                continue

            label_files = [f for f in os.listdir(src_label_dir) if f.endswith('.txt')]
            count = 0

            for label_file in label_files:
                label_path = os.path.join(src_label_dir, label_file)
                
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                
                # Kiểm tra sự tồn tại của class hiếm
                if any(line.split()[0] in RARE_CLASS_IDS for line in lines):
                    base_name = os.path.splitext(label_file)[0]
                    
                    # Tìm ảnh tương ứng
                    img_ext = None
                    for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
                        if os.path.exists(os.path.join(src_image_dir, base_name + ext)):
                            img_ext = ext
                            break
                    
                    if img_ext:
                        # Tạo tên file mới để tránh trùng lặp giữa các bộ dataset
                        # Ví dụ: ds0_image1.jpg, ds1_image1.jpg
                        new_name = f"ds{idx}_{base_name}"
                        
                        shutil.copy(
                            os.path.join(src_image_dir, base_name + img_ext),
                            os.path.join(OUTPUT_ROOT, subset, 'images', new_name + img_ext)
                        )
                        shutil.copy(
                            label_path,
                            os.path.join(OUTPUT_ROOT, subset, 'labels', new_name + '.txt')
                        )
                        count += 1
            
            print(f"   + [{subset}]: Đã lấy {count} ảnh có class hiếm.")

    print(f"\n✅ Hoàn tất! Dữ liệu đã được gộp tại: {OUTPUT_ROOT}")

if __name__ == "__main__":
    merge_and_filter_datasets()

In [ ]:
#gộp dataset
import os
import shutil

# Danh sách các dataset nguồn (đã được remap ở bước trước)
source_datasets = [
   # "/kaggle/working/dataset_2",
    #"/kaggle/working/dataset_3",
   # "/kaggle/working/merged_rare_dataset",
    "/kaggle/working/"
]

# Thư mục đích cuối cùng
target_root = "/kaggle/working/merged_dataset"

# Các tập con cần gộp
subsets = ['train', 'valid', 'test']

# Biến đếm tổng để đặt tên file không bị trùng (tránh ghi đè nếu file trùng tên)
img_id = 0

for subset in subsets:
    print(f"--- Đang gộp tập: {subset} ---")
    
    target_images = os.path.join(target_root, subset, "images")
    target_labels = os.path.join(target_root, subset, "labels")
    
    os.makedirs(target_images, exist_ok=True)
    os.makedirs(target_labels, exist_ok=True)

    for ds_path in source_datasets:
        img_folder = os.path.join(ds_path, subset, "images")
        label_folder = os.path.join(ds_path, subset, "labels")

        # Kiểm tra xem thư mục nguồn có tồn tại không (đề phòng dataset thiếu test/valid)
        if not os.path.exists(img_folder):
            continue

        files = os.listdir(img_folder)
        print(f"Processing: {img_folder} | Files: {len(files)}")

        for file in files:
            if file.lower().endswith((".jpg", ".png", ".jpeg")):
                img_path = os.path.join(img_folder, file)
                # Lấy tên file không có đuôi để tìm file label tương ứng
                file_name_no_ext = os.path.splitext(file)[0]
                label_path = os.path.join(label_folder, file_name_no_ext + ".txt")

                if os.path.exists(label_path):
                    # Đổi tên file để đảm bảo duy nhất: ví dụ train_0.jpg, train_1.jpg...
                    new_name = f"{subset}_{img_id}"
                    new_img_name = new_name + os.path.splitext(file)[1]
                    new_label_name = new_name + ".txt"

                    shutil.copy(img_path, os.path.join(target_images, new_img_name))
                    shutil.copy(label_path, os.path.join(target_labels, new_label_name))

                    img_id += 1
    
    print(f"Hoàn tất gộp tập {subset}. Tổng số ảnh hiện tại: {img_id}")

print("-" * 30)
print(f"TẤT CẢ ĐÃ XONG! Tổng cộng có {img_id} cặp ảnh/nhãn trong {target_root}")

In [ ]:
#7 cell bên dưới đây để thực hiện copy paste augmentation dữ liệu

In [ ]:
!pip install opencv-python-headless numpy tqdm Pillow -q

In [ ]:
# ===================== CẤU HÌNH =====================

# Đường dẫn dataset đầu vào (cấu trúc YOLOv8 chuẩn)
DATASET_DIR = "/kaggle/input/datasets/phandat123/my-vehicle-detection-datasets-v5/final_traffic_dataset"   # <-- Sửa đường dẫn này

# Đường dẫn output (cùng cấu trúc với input)
OUTPUT_DIR  = "/kaggle/working/"

# Chỉ augment split nào (val/test giữ nguyên)
AUGMENT_SPLITS     = ["test"]
PASSTHROUGH_SPLITS = [ "valid", "train"]   # copy nguyên, không chỉnh

# Tên class theo thứ tự trong data.yaml  <-- Sửa cho đúng
CLASS_NAMES = {
    0: "car",
    1: "bus",
    2: "truck",
    3: "motorcycle",
    4: "bicycle",
    5: "person"
}

# Class cần được tăng cường
RARE_CLASSES = [1, 2, 4]   # bus, truck, bicycle

# Số object tối đa paste thêm vào MỖI ảnh
MAX_PASTE_PER_IMAGE = 2

# True  → chỉ paste vào ảnh đang có rare class (an toàn hơn về context)
# False → paste vào tất cả ảnh train
ONLY_AUGMENT_IMAGES_WITH_RARE = True

# IoU tối đa cho phép khi tìm vị trí paste
MAX_IOU_THRESHOLD = 0.1

# Scale object khi paste so với kích thước gốc
SCALE_RANGE = (0.8, 1.2)

RANDOM_SEED = 42
# ====================================================

import os, random, shutil, warnings
import cv2, numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
warnings.filterwarnings('ignore')
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Cấu hình:")
print(f"  Input : {DATASET_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Rare classes: {[CLASS_NAMES[c] for c in RARE_CLASSES]}")
print(f"  Max paste/ảnh: {MAX_PASTE_PER_IMAGE}")

In [ ]:
def yolo_to_pixel(box, img_w, img_h):
    cx, cy, bw, bh = box
    x1 = int((cx - bw / 2) * img_w)
    y1 = int((cy - bh / 2) * img_h)
    x2 = int((cx + bw / 2) * img_w)
    y2 = int((cy + bh / 2) * img_h)
    return max(0, x1), max(0, y1), min(img_w, x2), min(img_h, y2)

def pixel_to_yolo(x1, y1, x2, y2, img_w, img_h):
    cx = ((x1 + x2) / 2) / img_w
    cy = ((y1 + y2) / 2) / img_h
    bw = (x2 - x1) / img_w
    bh = (y2 - y1) / img_h
    return cx, cy, bw, bh

def compute_iou(box1, box2):
    ix1 = max(box1[0], box2[0]); iy1 = max(box1[1], box2[1])
    ix2 = min(box1[2], box2[2]); iy2 = min(box1[3], box2[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0

def read_label(label_path):
    annotations = []
    p = Path(label_path)
    if not p.exists(): return annotations
    for line in p.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) == 5:
            annotations.append([int(parts[0])] + list(map(float, parts[1:])))
    return annotations

def write_label(label_path, annotations):
    with open(label_path, 'w') as f:
        for ann in annotations:
            f.write(f"{int(ann[0])} {ann[1]:.6f} {ann[2]:.6f} {ann[3]:.6f} {ann[4]:.6f}\n")

def extract_object(img, x1, y1, x2, y2):
    """Cắt object + tạo feathered mask để blend tự nhiên."""
    pad = 3
    x1p = max(0, x1-pad); y1p = max(0, y1-pad)
    x2p = min(img.shape[1], x2+pad); y2p = min(img.shape[0], y2+pad)
    crop = img[y1p:y2p, x1p:x2p].copy()
    h, w = crop.shape[:2]
    mask = np.ones((h, w), dtype=np.float32)
    feather = max(3, min(h, w) // 10)
    for i in range(feather):
        alpha = i / feather
        mask[i, :] = np.minimum(mask[i, :], alpha)
        mask[-(i+1), :] = np.minimum(mask[-(i+1), :], alpha)
        mask[:, i] = np.minimum(mask[:, i], alpha)
        mask[:, -(i+1)] = np.minimum(mask[:, -(i+1)], alpha)
    return crop, mask

def find_paste_position(img_h, img_w, obj_h, obj_w, existing_boxes, max_tries=60):
    """Tìm vị trí paste hợp lệ — không đè lên object cũ."""
    for _ in range(max_tries):
        x1 = random.randint(0, max(0, img_w - obj_w))
        y1 = random.randint(0, max(0, img_h - obj_h))
        x2, y2 = x1 + obj_w, y1 + obj_h
        if x2 > img_w or y2 > img_h: continue
        if all(compute_iou((x1,y1,x2,y2), eb) <= MAX_IOU_THRESHOLD
               for eb in existing_boxes):
            return x1, y1
    return None

def paste_onto(target, crop, mask, x1, y1):
    """Paste crop vào target tại (x1, y1) với alpha blending."""
    result = target.copy()
    h, w = crop.shape[:2]
    x2, y2 = min(x1+w, result.shape[1]), min(y1+h, result.shape[0])
    cw, ch = x2-x1, y2-y1
    if cw <= 0 or ch <= 0: return result
    a = mask[:ch, :cw, np.newaxis]
    result[y1:y2, x1:x2] = (
        a * crop[:ch, :cw] + (1-a) * result[y1:y2, x1:x2]
    ).astype(np.uint8)
    return result

print("✅ Utility functions sẵn sàng")

In [ ]:
def build_object_bank():
    """
    Quét split train, thu thập crop của tất cả object thuộc RARE_CLASSES.
    Chỉ lấy từ train để không leak val/test.
    """
    bank = {cls: [] for cls in RARE_CLASSES}
    img_dir = Path(DATASET_DIR) / "test" / "images"
    lbl_dir = Path(DATASET_DIR) / "test" / "labels"

    img_files = sorted(
        list(img_dir.glob("*.jpg")) +
        list(img_dir.glob("*.png")) +
        list(img_dir.glob("*.jpeg"))
    )

    print(f"Quét {len(img_files):,} ảnh train để build object bank...")
    skipped = 0

    for img_path in tqdm(img_files, desc="Building bank"):
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        anns = read_label(lbl_path)
        rare = [a for a in anns if a[0] in RARE_CLASSES]
        if not rare: continue

        img = cv2.imread(str(img_path))
        if img is None: skipped += 1; continue
        h, w = img.shape[:2]

        for ann in rare:
            cls_id = int(ann[0])
            x1, y1, x2, y2 = yolo_to_pixel(ann[1:], w, h)
            if (x2-x1) < 20 or (y2-y1) < 20: continue
            crop, mask = extract_object(img, x1, y1, x2, y2)
            bank[cls_id].append((crop, mask, x2-x1, y2-y1))

    print(f"\n📦 Object Bank (bỏ qua {skipped} ảnh lỗi):")
    for cls_id, objs in bank.items():
        print(f"   {CLASS_NAMES.get(cls_id, cls_id):<12}: {len(objs):,} objects")
    return bank


object_bank = build_object_bank()

In [ ]:
def process_image(img_path, src_lbl_dir, out_img_dir, out_lbl_dir):
    """
    Đọc ảnh từ input, paste thêm object nếu đủ điều kiện,
    ghi ra output với TÊN FILE GIỮ NGUYÊN.
    Trả về số object đã paste được.
    """
    lbl_path = src_lbl_dir / (img_path.stem + ".txt")
    anns = read_label(lbl_path)

    # Kiểm tra điều kiện: có rare class không?
    if ONLY_AUGMENT_IMAGES_WITH_RARE:
        has_rare = any(a[0] in RARE_CLASSES for a in anns)
        if not has_rare:
            # Không augment — copy thẳng sang output
            shutil.copy2(img_path, out_img_dir / img_path.name)
            dst_lbl = out_lbl_dir / (img_path.stem + ".txt")
            if lbl_path.exists():
                shutil.copy2(lbl_path, dst_lbl)
            else:
                dst_lbl.write_text("")
            return 0

    img = cv2.imread(str(img_path))
    if img is None:
        return 0

    img_h, img_w = img.shape[:2]
    existing_boxes = [yolo_to_pixel(a[1:], img_w, img_h) for a in anns]
    new_anns = anns.copy()
    result   = img.copy()
    pasted   = 0

    classes_to_try = [c for c in RARE_CLASSES if object_bank[c]]
    random.shuffle(classes_to_try)

    for cls_id in classes_to_try:
        if pasted >= MAX_PASTE_PER_IMAGE:
            break

        crop, mask, orig_w, orig_h = random.choice(object_bank[cls_id])

        scale = random.uniform(*SCALE_RANGE)
        new_w = max(20, int(orig_w * scale))
        new_h = max(20, int(orig_h * scale))

        # Bỏ qua nếu object quá to so với ảnh
        if new_w > img_w * 0.6 or new_h > img_h * 0.6:
            continue

        pos = find_paste_position(img_h, img_w, new_h, new_w, existing_boxes)
        if pos is None:
            continue

        px1, py1 = pos
        px2, py2 = px1 + new_w, py1 + new_h

        rc = cv2.resize(crop, (new_w, new_h))
        rm = cv2.resize(mask, (new_w, new_h))

        result = paste_onto(result, rc, rm, px1, py1)

        cx, cy, bw, bh = pixel_to_yolo(px1, py1, px2, py2, img_w, img_h)
        new_anns.append([cls_id, cx, cy, bw, bh])
        existing_boxes.append((px1, py1, px2, py2))
        pasted += 1

    # Ghi ra output — tên file GIỐNG HỆT input
    cv2.imwrite(str(out_img_dir / img_path.name), result)
    write_label(out_lbl_dir / (img_path.stem + ".txt"), new_anns)
    return pasted


print("✅ process_image sẵn sàng")

In [ ]:
def run():
    out_root = Path(OUTPUT_DIR)

    # --- Augment splits ---
    for split in AUGMENT_SPLITS:
        src_img = Path(DATASET_DIR) / split / "images" 
        src_lbl = Path(DATASET_DIR) / split / "labels"
        out_img = out_root / split / "images"
        out_lbl = out_root / split / "labels"
        out_img.mkdir(parents=True, exist_ok=True)
        out_lbl.mkdir(parents=True, exist_ok=True)

        if not src_img.exists():
            print(f"⚠️  Không tìm thấy {src_img}, bỏ qua"); continue

        img_files = sorted(
            list(src_img.glob("*.jpg")) +
            list(src_img.glob("*.png")) +
            list(src_img.glob("*.jpeg"))
        )

        print(f"\n🔄 Augmenting '{split}' — {len(img_files):,} ảnh")

        total_pasted   = 0
        augmented_count = 0
        for img_path in tqdm(img_files, desc=f"{split}"):
            n = process_image(img_path, src_lbl, out_img, out_lbl)
            total_pasted   += n
            if n > 0: augmented_count += 1

        n_out = len(list(out_img.iterdir()))
        print(f"   ✅ {augmented_count:,}/{len(img_files):,} ảnh được augment")
        print(f"   ✅ Tổng objects đã paste thêm: {total_pasted:,}")
        print(f"   ✅ Input: {len(img_files):,} ảnh  →  Output: {n_out:,} ảnh")

    # --- Passthrough splits (copy nguyên không chỉnh) ---
    for split in PASSTHROUGH_SPLITS:
        src_img = Path(DATASET_DIR) / split / "images"
        src_lbl = Path(DATASET_DIR) / split / "labels"
        out_img = out_root / split / "images"
        out_lbl = out_root / split / "labels"
        if not src_img.exists(): continue
        out_img.mkdir(parents=True, exist_ok=True)
        out_lbl.mkdir(parents=True, exist_ok=True)
        for f in src_img.iterdir(): shutil.copy2(f, out_img / f.name)
        for f in src_lbl.iterdir(): shutil.copy2(f, out_lbl / f.name)
        print(f"\n📋 '{split}' — copy nguyên {len(list(src_img.iterdir())):,} ảnh")

    # Copy data.yaml
    for yaml_name in ["data.yaml", "dataset.yaml"]:
        yaml_src = Path(DATASET_DIR) / yaml_name
        if yaml_src.exists():
            shutil.copy2(yaml_src, out_root / yaml_name)
            print(f"\n📄 {yaml_name} đã copy sang output")
            break

    print(f"\n🎉 Hoàn thành! Output: {out_root.absolute()}")


run()

In [ ]:
def count_classes(dataset_dir, splits=("train",)):
    counts = defaultdict(int)
    n_images = 0
    for split in splits:
        lbl_dir = Path(dataset_dir) / "labels" / split
        if not lbl_dir.exists(): continue
        files = list(lbl_dir.glob("*.txt"))
        n_images += len(files)
        for f in files:
            for ann in read_label(f):
                counts[int(ann[0])] += 1
    return counts, n_images

before, n_before = count_classes(DATASET_DIR)
after,  n_after  = count_classes(OUTPUT_DIR)

total_b = sum(before.values())
total_a = sum(after.values())

print(f"{'Class':<14} {'Trước':>10} {'Sau':>10} {'Tăng':>8} {'%Trước':>8} {'%Sau':>8}")
print("-" * 62)
for cls_id in sorted(set(before) | set(after)):
    name = CLASS_NAMES.get(cls_id, f"cls{cls_id}")
    b    = before.get(cls_id, 0)
    a    = after.get(cls_id, 0)
    diff = a - b
    pb   = b / total_b * 100 if total_b else 0
    pa   = a / total_a * 100 if total_a else 0
    mark = " ⬆️" if cls_id in RARE_CLASSES and diff > 0 else ""
    print(f"{name:<14} {b:>10,} {a:>10,} {diff:>+8,} {pb:>7.2f}% {pa:>7.2f}%{mark}")

print("-" * 62)
print(f"{'TOTAL':<14} {total_b:>10,} {total_a:>10,} {total_a-total_b:>+8,}")
print(f"\nSố ảnh train — Input: {n_before:,}  |  Output: {n_after:,}")

# Assertion đảm bảo số ảnh không thay đổi
assert n_before == n_after, f"❌ Số ảnh thay đổi! {n_before} → {n_after}"
print("✅ Số lượng ảnh khớp hoàn toàn")

In [ ]:
import shutil
# Nén thư mục dataset đã lọc thành file zip
shutil.make_archive('final_traffic_dataset', 'zip', '/kaggle/working/merged_dataset')
print("Nén xong! Bạn hãy Refresh cột bên phải và tải file final_traffic_dataset.zip về máy.")

In [ ]:
# Xóa thư mục và tất cả nội dung bên trong (-r là recursive, -f là force)
!rm -rf /kaggle/working/runs/detect/val-4

In [ ]:
!rm -f /kaggle/working/dataset-metadata.json
!kaggle datasets init -p /kaggle/working

In [ ]:
import json

meta_path = "/kaggle/working/dataset-metadata.json"

meta = {
    "title": "My-vehicle-detection-datasets-v5",
    "id": "phandat123/my-vehicle-detection-datasets-v5",
    "licenses": [{"name": "CC0-1.0"}]
}

with open(meta_path, "w") as f:
    json.dump(meta, f)

print("Metadata updated")

In [ ]:
!kaggle datasets create -p /kaggle/working

In [ ]:
yaml_content = """
path: /kaggle/working/

train: train/images
val: valid/images
test: test/images
nc: 6

names:
  0: car
  1: bus
  2: truck
  3: motorcycle
  4: bicycle
  5: person
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(yaml_content)

In [ ]:
# check format yolo with datasets

label_dir = "/kaggle/input/datasets/phandat123/my-vehicle-detection-dataset/final_traffic_dataset/test/labels"

for file in os.listdir(label_dir):
    with open(os.path.join(label_dir, file)) as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            if len(parts) != 5:
                print("Lỗi format:", file)
            if any(x < 0 or x > 1 for x in parts[1:]):
                print("Out of range:", file)

In [ ]:
# chỉnh lại format 
import os
import shutil

dataset_root = "/kaggle/input/datasets/phandat123/my-vehicle-detection-dataset/final_traffic_dataset"
output_root = "/kaggle/working/clean_dataset"

splits = ["train", "valid", "test"]

for split in splits:
    print(f"\n🔍 Processing {split}...")

    img_src = os.path.join(dataset_root, split, "images")
    label_src = os.path.join(dataset_root, split, "labels")

    img_dst = os.path.join(output_root, split, "images")
    label_dst = os.path.join(output_root, split, "labels")

    os.makedirs(img_dst, exist_ok=True)
    os.makedirs(label_dst, exist_ok=True)

    # 1. Copy ảnh (giữ nguyên)
    for file in os.listdir(img_src):
        shutil.copy(os.path.join(img_src, file), os.path.join(img_dst, file))

    # 2. Xử lý và lọc label
    count_removed = 0
    for file in os.listdir(label_src):
        src_path = os.path.join(label_src, file)
        dst_path = os.path.join(label_dst, file)

        new_lines = []
        with open(src_path, "r") as f:
            for line in f:
                parts = line.strip().split()

                # Kiểm tra: Phải có ĐÚNG 5 phần tử
                if len(parts) == 5:
                    try:
                        # Kiểm tra thêm: Các giá trị tọa độ phải là số thực
                        # (Bước này giúp tránh lỗi nếu file label chứa ký tự lạ)
                        values = [float(x) for x in parts]
                        new_lines.append(" ".join(parts))
                    except ValueError:
                        count_removed += 1
                else:
                    # Nếu > 5 hoặc < 5 phần tử thì bỏ qua (xóa dòng đó)
                    count_removed += 1

        # Ghi lại file label mới (chỉ chứa các dòng hợp lệ)
        with open(dst_path, "w") as f:
            f.write("\n".join(new_lines))
    
    if count_removed > 0:
        print(f"⚠️ Đã loại bỏ {count_removed} dòng sai format trong tập {split}.")

print("\n✅ Done! Dataset sạch đã sẵn sàng tại /kaggle/working/clean_dataset")

In [ ]:
 model.train(
    data="/kaggle/working/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001
)


In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/input/models/phandat123/model-v2m/pytorch/default/1/best_v2m.pt")

metrics = model.val(data="/kaggle/working/data.yaml", split="test")